# DalMax — results doctor (verify & migrate ablation trees)

Runnable companion to the `results_doctor` CLI
(`dalmax/reporting/results_doctor.py`, `python -m dalmax.reporting.results_doctor`).
See [`COLAB_RUNBOOK.md`](https://github.com/MarioCarvalhoBr/dalmax-deep-active-learning-python/blob/main/COLAB_RUNBOOK.md)'s "Verifying and migrating existing results" section for the full write-up — this notebook
is the runnable form of it.

**What this notebook does:**
1. **Verifies** an `results/ablations/` tree against the expected
   `(study, config, seed)` grid for one or both ablation suites (`rnhal`/`texhal`),
   reporting OK / INCOMPLETE / MISSING per triple.
2. **Migrates**, once, the already-executed legacy RNHAL tree
   (`results/ablations/{6_1,6_2,6_3}`, no method segment) into the per-method
   layout (`results/ablations/rnhal/...`) — a `results/`-internal MOVE, never an
   edit or delete (`.claude/rules/data-safety.md`).

**This is a CPU-only notebook** — no GPU/training happens here, only filesystem
checks and (optionally) directory moves. It works both on **Colab** (after Drive
mount + repo setup, sections 1-3 below) and on the **lab machine or local checkout**
(skip straight to section 4).

In [ ]:
import os

# Parameters -- the ONLY cell you should normally need to edit.
DRIVE_ROOT = "/content/drive/MyDrive/UFMS/Pós-graduação/Doutorado/FINAL/PROJETO/DALMAX"
REPO_URL = "https://github.com/MarioCarvalhoBr/dalmax-deep-active-learning-python.git"
REPO_DIR = "/content/dalmax"
BRANCH = "main"
METHOD = "all"  # "rnhal" | "texhal" | "all" -- which suite(s) to verify (see
               # files_config/ablations/README.md)
APPLY_MIGRATION = False  # explicit opt-in -- leave False to dry-run migrate-legacy;
                         # set True only once you have reviewed the dry-run output
                         # below and are ready to actually move the legacy tree.

# Also export as env vars so every `!`-shell cell below sees them too.
os.environ["DRIVE_ROOT"] = DRIVE_ROOT
os.environ["METHOD"] = METHOD

print("DRIVE_ROOT:      ", DRIVE_ROOT)
print("REPO_DIR:        ", REPO_DIR)
print("METHOD:          ", METHOD)
print("APPLY_MIGRATION: ", APPLY_MIGRATION)

## 1. Colab-only setup

**Skip these two cells if you are running locally or on the lab machine** — jump
straight to section 3 (Wire results). These two cells mount Drive and clone/pull +
install the repo, exactly like [`notebooks/colab_runbook.ipynb`](colab_runbook.ipynb).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import subprocess

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"{REPO_DIR} already exists -- pulling latest {BRANCH}...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
else:
    print(f"Cloning {REPO_URL} (branch {BRANCH}) into {REPO_DIR}...")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True
    )

os.chdir(REPO_DIR)
%cd $REPO_DIR
!git log --oneline -1

Verification only needs the repo's code (no torch/GPU work happens in this
notebook) — if a `.venv` already exists from an earlier session this is fast;
otherwise `poetry install` still downloads the full pinned dependency set (including
the CUDA wheels), same as any other Colab session.

In [ ]:
!pip install -q poetry
!poetry install

## 2. Wire up `results/` (Drive)

On Colab, `results/` must point at Drive so `verify`/`migrate-legacy` see the real
tree instead of an empty local directory. `make colab-setup` (same target as
[`colab_runbook.ipynb`](colab_runbook.ipynb)) is idempotent — safe to re-run even if
the symlink is already in place. **On the lab machine or local checkout, skip this
cell** — `results/` is already the real directory there.

In [ ]:
!make colab-setup

## 3. Migrate the legacy RNHAL tree (one-time)

The already-executed 33-run RNHAL batch (2026-08-25/26) lives at the LEGACY
no-method-segment root `results/ablations/{6_1,6_2,6_3}/` (see `CLAUDE.md`'s
"Per-method ablation layout" note). `migrate-legacy` moves it into the per-method
layout (`results/ablations/rnhal/...`) so it lines up with `METHOD=rnhal`'s
`SKIP_EXISTING` glob and `make ablation-report METHOD=rnhal`.

**No-overwrite guarantee**: if any destination already exists, that item is skipped
with a `CONFLICT` message and the command exits nonzero — it never overwrites or
deletes anything. This is a `results/`-internal **move** (file bytes untouched, only
the parent directory changes), not an edit — consistent with
`.claude/rules/data-safety.md`'s append-only policy.

The cell below always runs a dry run first so you can review the exact moves before
anything happens.

In [ ]:
!poetry run python -m dalmax.reporting.results_doctor migrate-legacy --root results/ablations

Review the moves printed above. If they look right, set `APPLY_MIGRATION = True` in
the Parameters cell and re-run it, then run the guarded cell below to actually
perform the migration.

In [ ]:
if APPLY_MIGRATION:
    !poetry run python -m dalmax.reporting.results_doctor migrate-legacy --root results/ablations --apply
else:
    print("dry-run only -- set APPLY_MIGRATION=True and re-run to actually migrate.")

## 4. Verify the results tree

Checks every expected `(study, config, seed)` triple for `METHOD` against the full
artifact set a successful run writes (`results.json`, `predictions.csv`,
`run_metadata.json`, plots, `log-dalmax.log`, `saved_model.pth`). For `rnhal` this
also transparently checks the legacy layout, so **you do not need to migrate before
verifying** — a triple satisfied in either layout reports OK.

In [ ]:
!poetry run python -m dalmax.reporting.results_doctor verify --root results/ablations --method {METHOD}

**Interpreting the statuses:**
- **OK** (current layout) or **OK (legacy layout)** — nothing to do.
- **INCOMPLETE** — the leaf directory exists but is missing one or more required
  files (listed inline). Re-run just that triple with the single-run CLI pattern
  (see `LAB_RUNBOOK.md` §3 / `COLAB_RUNBOOK.md` §6) — `SKIP_EXISTING=1` will keep
  every already-completed triple untouched and only redo the incomplete/missing
  ones.
- **MISSING** — the triple has not been run at all yet. Same fix: run it (or the
  whole batch) with the ablation scripts; `SKIP_EXISTING` makes this safe to
  relaunch repeatedly.

The command exits 0 only when every expected triple is OK (in any layout); a
nonzero exit means at least one triple is INCOMPLETE or MISSING.

## 5. Optional: aggregate into report tables

Run these once the batch you care about is fully OK per section 4 above (after
migration, `rnhal`'s tables come from the per-method root; `ablation-report-legacy`
remains available for the legacy root directly).

In [ ]:
!make ablation-report METHOD=rnhal

In [ ]:
!make ablation-report METHOD=texhal